In [23]:
from pathlib import Path

In [24]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import gc
import time
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

import torch

In [25]:
%run universal-framework.ipynb

In [12]:
import chronos
import transformers

print("chronos:", chronos.__version__)
print("transformers:", transformers.__version__)

chronos: 2.3.1
transformers: 5.14.1


In [13]:
from chronos import ChronosPipeline

chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="auto"
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [22]:
def chronos_forecast(train, horizon):

    context = torch.tensor(
        train,
        dtype=torch.float32
    )

    forecast = chronos_model.predict(
        context,
        prediction_length=horizon
    )

    forecast = forecast.cpu().numpy()

    return np.median(
        forecast[0],
        axis=0
    )

In [ ]:
chronos_results = []

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    print(f"\nRunning Chronos on {dataset_name}")

    df = pd.read_csv(
        file,
        low_memory=False
    )

    for horizon, horizon_type in get_horizons_info(
        dataset_name
    ):

        result = benchmark_dataset(
            df=df,
            horizon=horizon,
            forecast_function=chronos_forecast,
            model_name="Chronos"
        )

        if result is not None:

            result["dataset"] = dataset_name
            result["horizon_type"] = horizon_type

            chronos_results.append(result)

            pd.DataFrame(
                chronos_results
            ).to_csv(
                RESULTS_DIR / "chronos.csv",
                index=False
            )

    del df
    gc.collect()

chronos_results = pd.DataFrame(
    chronos_results
)


Running Chronos on std_agri_medium

Running Chronos on std_agri_small

Running Chronos on std_energy_medium


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

In [1]:
import lag_llama

In [7]:
from itertools import islice



from matplotlib import pyplot as plt

import matplotlib.dates as mdates

from tqdm.autonotebook import tqdm



import torch

from gluonts.evaluation import make_evaluation_predictions, Evaluator

from gluonts.dataset.repository.datasets import get_dataset



from gluonts.dataset.pandas import PandasDataset

import pandas as pd



from lag_llama.gluon.estimator import LagLlamaEstimator

AttributeError: partially initialized module 'pytorch_lightning' has no attribute 'accelerators' (most likely due to a circular import)

In [1]:
from Lag_Llama import LagLlama

lag_llama_model = LagLlama.load_from_checkpoint(
    "lag-llama.ckpt"
)

ModuleNotFoundError: No module named 'Lag_Llama'

In [ ]:
def lagllama_forecast(
    train,
    horizon
):

    forecast = lag_llama_model.predict(
        train,
        prediction_length=horizon
    )

    return np.asarray(forecast)

In [ ]:
lagllama_results = []

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    print(f"\nRunning Lag-Llama on {dataset_name}")

    df = pd.read_csv(
        file,
        low_memory=False
    )

    for horizon, horizon_type in get_horizons(
        dataset_name
    ):

        result = benchmark_dataset(
            df=df,
            horizon=horizon,
            forecast_function=lagllama_forecast,
            model_name="LagLlama"
        )

        if result is not None:

            result["dataset"] = dataset_name
            result["horizon_type"] = horizon_type

            lagllama_results.append(result)

            pd.DataFrame(
                lagllama_results
            ).to_csv(
                RESULTS_DIR / "lagllama.csv",
                index=False
            )

    del df
    gc.collect()

lagllama_results = pd.DataFrame(
    lagllama_results)

In [26]:
import os
import math
import tempfile
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments, set_seed
import numpy as np
import pandas as pd

# TSFM libraries
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
from tsfm_public.toolkit.callbacks import TrackingCallback

In [27]:
from tsfm_public import (
    TimeSeriesPreprocessor,
    TinyTimeMixerForPrediction,
    TrackingCallback,
    count_parameters,
    get_datasets,
)


In [28]:
#from transformers import Automodel
import transformers

ttm_model = TinyTimeMixerForPrediction.from_pretrained(
    "ibm-research/ttm-r3"
)

Some weights of TinyTimeMixerForPrediction were not initialized from the model checkpoint at ibm-research/ttm-r3 and are newly initialized: ['backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.gating_block.attn_layer.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.gating_block.attn_layer.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc1.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc1.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc2.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.mlp.fc2.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.norm.norm.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.feature_mixer.norm.norm.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.patch_mixer.gating_block.attn_layer.bias', 'backbone.encoder.mlp_mixer_encoder.mixers.0.patch_mixer.gating_block.attn_layer.weight', 'backbone.encoder.mlp_mixer_encoder.mixers.0.

In [36]:

CONTEXT_LENGTH = 512
MODEL_HORIZON = 30

def ttm_forecast(
    train,
    horizon
):

    history = np.asarray(
        train,
        dtype=np.float32
    ).copy()

    forecasts = []

    while len(forecasts) < horizon:

        if len(history) >= CONTEXT_LENGTH:
            context = history[-CONTEXT_LENGTH:]
        else:
            pad = np.full(
                CONTEXT_LENGTH - len(history),
                history[0],
                dtype=np.float32
            )
            context = np.concatenate(
                [pad, history]
            )

        context = torch.tensor(
            context,
            dtype=torch.float32
        ).reshape(
            1,
            CONTEXT_LENGTH,
            1
        )

        with torch.no_grad():

            outputs = ttm_model(
                past_values=context
            )

        chunk = (
            outputs.prediction_outputs
            .squeeze()
            .cpu()
            .numpy()
        )

        forecasts.extend(
            chunk.tolist()
        )

        history = np.concatenate(
            [history, chunk]
        )

    return np.asarray(
        forecasts[:horizon]
    )

In [37]:
print(ttm_model.config.prediction_length)

30


In [38]:
ttm_results = []

for file in DATA_DIR.glob("*.csv"):

    dataset_name = file.stem

    print(f"\nRunning TTM-1 on {dataset_name}")

    df = pd.read_csv(
        file,
        low_memory=False
    )

    for horizon, horizon_type in get_horizons_info(
        dataset_name
    ):

        result = benchmark_dataset(
            df=df,
            horizon=horizon,
            forecast_function=ttm_forecast,
            model_name="TTM-1"
        )

        if result is not None:

            result["dataset"] = dataset_name
            result["horizon_type"] = horizon_type

            ttm_results.append(result)

            pd.DataFrame(
                ttm_results
            ).to_csv(
                RESULTS_DIR / "ttm1.csv",
                index=False
            )

    del df
    gc.collect()

ttm_results = pd.DataFrame(
    ttm_results
)


Running TTM-1 on std_agri_medium

Running TTM-1 on std_agri_small

Running TTM-1 on std_energy_medium

Running TTM-1 on std_energy_small

Running TTM-1 on std_favorita_medium_train

Running TTM-1 on std_favorita_medium_transactions

Running TTM-1 on std_favorita_small_train

Running TTM-1 on std_favorita_small_transactions

Running TTM-1 on std_m4_daily_medium

Running TTM-1 on std_m4_daily_small

Running TTM-1 on std_m4_hourly_medium

Running TTM-1 on std_m4_hourly_small

Running TTM-1 on std_m4_monthly_medium

Running TTM-1 on std_m4_monthly_small

Running TTM-1 on std_m4_quarterly_medium

Running TTM-1 on std_m4_quarterly_small

Running TTM-1 on std_m4_weekly_medium

Running TTM-1 on std_m4_weekly_small

Running TTM-1 on std_m4_yearly_medium

Running TTM-1 on std_m4_yearly_small

Running TTM-1 on std_traffic_medium

Running TTM-1 on std_traffic_small


In [40]:
foundation_results = pd.concat(
    [
        #chronos_results,
        #lagllama_results,
        ttm_results
    ],
    ignore_index=True
)

foundation_results.to_csv(
    RESULTS_DIR / "foundation_models.csv",
    index=False
)

foundation_results

,model,n_series,n_rows,horizon,runtime_seconds,MAE,RMSE,SMAPE,dataset,horizon_type
0,TTM-1,128,7791,1,4.228150,1.873994e+06,1.873994e+06,107.251085,std_agri_medium,short
1,TTM-1,128,7791,3,4.377415,1.814837e+06,1.820458e+06,106.709883,std_agri_medium,medium
2,TTM-1,128,7791,5,3.887839,1.758306e+06,1.766346e+06,106.516976,std_agri_medium,long
3,TTM-1,27,1560,1,0.874799,2.458460e+06,2.458460e+06,101.291424,std_agri_small,short
4,TTM-1,27,1560,3,1.226065,2.389375e+06,2.393738e+06,101.277372,std_agri_small,medium
...,...,...,...,...,...,...,...,...,...,...
61,TTM-1,207,3547152,24,4.430707,6.236625e+00,6.825810e+00,12.258931,std_traffic_medium,medium
62,TTM-1,207,3547152,48,7.763869,6.772750e+00,7.381212e+00,12.843492,std_traffic_medium,long
63,TTM-1,207,709389,12,3.792797,5.353289e+00,5.636999e+00,9.930250,std_traffic_small,short
64,TTM-1,207,709389,24,3.845109,5.063876e+00,5.418067e+00,9.383839,std_traffic_small,medium
